In [1]:
from kaggle_secrets import UserSecretsClient
import wandb

# Initialize the client to access secrets
user_secrets = UserSecretsClient()

# Get the key you just stored
api_key = user_secrets.get_secret("WANDB_API_KEY") 

# Log in to wandb
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saurabh200206 (saurabh200206-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import torch
import joblib

2025-09-12 01:46:26.439966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757641586.734379      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757641586.819645      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
MODEL_NAME = '/kaggle/input/finalmodel/final-deberta-v3-model' 
N_SPLITS = 5
RANDOM_STATE = 42
MAX_LENGTH = 512
BATCH_SIZE = 8

In [4]:
print("Loading and preparing data...")
train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")

# Combine text fields into a single input
train_df['input_text'] = train_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}",
    axis=1
)

train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']

Loading and preparing data...


In [5]:
print("Loading pre-fitted MultiLabelBinarizer...")
mlb = joblib.load("/kaggle/input/finalmodel/mlb.joblib")
labels = mlb.classes_
num_labels = len(labels)
print(f"Total unique labels: {num_labels}")

Loading pre-fitted MultiLabelBinarizer...
Total unique labels: 65


In [6]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
y_stratify = pd.Categorical(train_df['full_label']).codes

In [7]:
train_df['oof_prediction_top1'] = ""
oof_predictions = np.zeros((len(train_df), num_labels)) # To store logits/probabilities
mod ="/kaggle/input/deberta-v3-base-offline-files/deberta-v3-base-offline"

# Load tokenizer once (from the same fine-tuned model directory)
tokenizer = AutoTokenizer.from_pretrained(mod)

In [8]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)


In [9]:
for fold_num, (train_idx, val_idx) in enumerate(skf.split(train_df, y_stratify)):
    print("-" * 50)
    print(f"--- Starting Fold {fold_num + 1}/{N_SPLITS} ---")
    print("-" * 50)
    
    fold_train_df = train_df.iloc[train_idx]
    fold_val_df = train_df.iloc[val_idx]

    fold_train_labels = mlb.transform(fold_train_df['full_label'].apply(lambda x: [x])).astype(np.float32)
    fold_val_labels = mlb.transform(fold_val_df['full_label'].apply(lambda x: [x])).astype(np.float32)

    train_dataset = Dataset.from_dict({'text': fold_train_df['input_text'].tolist(), 'labels': fold_train_labels})
    val_dataset = Dataset.from_dict({'text': fold_val_df['input_text'].tolist(), 'labels': fold_val_labels})

    train_tokenized = train_dataset.map(tokenize_function, batched=True)
    val_tokenized = val_dataset.map(tokenize_function, batched=True)
    train_tokenized.set_format("torch")
    val_tokenized.set_format("torch")

    print(f"Loading fine-tuned model from local path: {MODEL_NAME}...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        problem_type="multi_label_classification",
        id2label={i: label for i, label in enumerate(labels)},
        label2id={label: i for i, label in enumerate(labels)}
    )

    training_args = TrainingArguments(
        output_dir=f'./results_fold_{fold_num}',
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        logging_steps=500,
        eval_strategy="no",
        save_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
    )
    print("Training model for this fold...")
    trainer.train()

    print("Making predictions on validation set...")
    predictions = trainer.predict(val_tokenized)
    fold_logits = predictions.predictions
    
    oof_predictions[val_idx] = fold_logits

    top1_indices = np.argmax(fold_logits, axis=1)
    top1_labels = mlb.classes_[top1_indices]
    
    train_df.loc[val_idx, 'oof_prediction_top1'] = top1_labels

    del model, trainer, train_dataset, val_dataset, train_tokenized, val_tokenized
    torch.cuda.empty_cache()

print("\nCross-validation complete.")
output_df = train_df[['full_label', 'oof_prediction_top1']].copy()
output_df.rename(columns={'full_label': 'true_label', 'oof_prediction_top1': 'predicted_label_top1'}, inplace=True)

output_df.to_csv('oof_predictions.csv', index=False)
print("OOF predictions saved to 'oof_predictions.csv'")

np.save('oof_raw_predictions.npy', oof_predictions)
print("Raw OOF logits saved to 'oof_raw_predictions.npy'")


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


--------------------------------------------------
--- Starting Fold 1/5 ---
--------------------------------------------------


Map:   0%|          | 0/29356 [00:00<?, ? examples/s]

Map:   0%|          | 0/7340 [00:00<?, ? examples/s]

Loading fine-tuned model from local path: /kaggle/input/finalmodel/final-deberta-v3-model...


...

Loading fine-tuned model from local path: /kaggle/input/finalmodel/final-deberta-v3-model...
Training model for this fold...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,0.014400
1000,0.013000
1500,0.011600


Making predictions on validation set...



Cross-validation complete.
OOF predictions saved to 'oof_predictions.csv'
Raw OOF logits saved to 'oof_raw_predictions.npy'
